In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

# ==========================================
# 1. Configuration (Matches Your Notebook)
# ==========================================
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2  
IMG_W = IMG_H        = 1024

# Update these paths to match your current run environment
MODEL_WEIGHTS_PATH   = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth" 
TEST_IMAGE_DIR       = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set"
RAW_SUBMISSION_CSV   = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/sample_submission.csv" # The file containing raw inferences
CLEANED_OUTPUT_CSV   = "submission.csv"
# ==========================================
# 2. Helper Functions
# ==========================================

def setup_detectron_predictor(weights_path):
    """Initializes the Detectron2 predictor using your precise architecture settings."""
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights_path
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
    return DefaultPredictor(cfg)

def load_for_inference(path):
    """Your notebook's exact 16-bit to float32 image loading pipeline."""
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        return None
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def add_gaussian_noise(image, mean=0, std=15):
    """Adds Gaussian noise directly to the float32 processed image."""
    noise = np.random.normal(mean, std, image.shape).astype(np.float32)
    noisy_image = image + noise
    return np.clip(noisy_image, 0, 255).astype(np.float32)

def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union (IoU) for two boxes [x1, y1, x2, y2]."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-6)

def parse_prediction_string(pred_string):
    """Parses 'conf x y w h' string into structured lists of boxes [x1,y1,x2,y2] and confs."""
    if pd.isna(pred_string) or pred_string.strip() == "":
        return []
    
    parts = pred_string.strip().split()
    detections = []
    for i in range(0, len(parts), 5):
        conf = float(parts[i])
        x = float(parts[i+1])
        y = float(parts[i+2])
        w = float(parts[i+3])
        h = float(parts[i+4])
        detections.append({
            'bbox': [x, y, x + w, y + h],
            'conf': conf
        })
    return detections

# ==========================================
# 3. Calibration Logic
# ==========================================

def calibrate_pipeline(df, predictor, num_samples=5, target_delta=0.2, iou_thresh=0.5):
    """
    Sweeps noise ranges on a small subset of data to automatically find the ideal noise 
    level and decide whether to keep stable or fluctuating streaks.
    """
    print(f"\n--- Starting Calibration Step (Sampling {num_samples} images) ---")
    
    # Filter rows that actually contain predictions to run diagnostics on
    valid_rows = df[df['prediction_string'].notna() & (df['prediction_string'].str.strip() != "")].head(num_samples)
    
    if len(valid_rows) == 0:
        print("Warning: No predictions found in submission file to calibrate with. Using defaults.")
        return 20.0, True # Default noise std=20, keep_stable=True
        
    candidate_stds = [5, 10, 15, 20, 25, 30, 40]
    best_std = 20.0
    min_error = float('inf')
    
    # Track shifts at a high standard noise (std=25) to test if the model is rigidly stubborn (Stable Poison)
    shifts_at_diagnostic_noise = []
    
    for std in candidate_stds:
        all_deltas = []
        
        for _, row in valid_rows.iterrows():
            img_path = Path(TEST_IMAGE_DIR) / f"{row['image_id']}.png"
            image = load_for_inference(img_path)
            if image is None: continue
                
            orig_streaks = parse_prediction_string(row['prediction_string'])
            noisy_image = add_gaussian_noise(image, std=std)
            
            outputs = predictor(noisy_image)["instances"].to("cpu")
            noisy_boxes = outputs.pred_boxes.tensor.numpy()
            noisy_scores = outputs.scores.numpy()
            new_streaks = [{'bbox': b, 'conf': s} for b, s in zip(noisy_boxes, noisy_scores)]
            
            for orig in orig_streaks:
                best_iou = 0.0
                matched_conf = 0.0
                for new_strk in new_streaks:
                    iou = calculate_iou(orig['bbox'], new_strk['bbox'])
                    if iou > best_iou:
                        best_iou = iou
                        matched_conf = new_strk['conf']
                        
                if best_iou < iou_thresh:
                    matched_conf = 0.0
                    
                all_deltas.append(abs(orig['conf'] - matched_conf))
                
        if not all_deltas: continue
        
        mean_delta = np.mean(all_deltas)
        error = abs(mean_delta - target_delta)
        print(f"-> Tested Noise STD {std:2d} | Average Confidence Shift: {mean_delta:.4f}")
        
        if error < min_error:
            min_error = error
            best_std = std
            
        if std == 25:
            shifts_at_diagnostic_noise = all_deltas

    # Determine if poison is highly stable (stubborn) or fluctuating
    # If even under high noise (std=25), the median shift is tiny (< 0.1), the poison is over-indexed/stable.
    median_diagnostic_shift = np.median(shifts_at_diagnostic_noise) if shifts_at_diagnostic_noise else 0.2
    
    if median_diagnostic_shift < 0.12:
        print(f"\n[Decision] Median shift at high noise is very low ({median_diagnostic_shift:.4f}).")
        print("👉 POISON IS STABLE (Hyper-memorized). Target Strategy: KEEP FLUCTUATING STREAKS.")
        keep_stable = False
    else:
        print(f"\n[Decision] Median shift at high noise is noticeable ({median_diagnostic_shift:.4f}).")
        print("👉 POISON IS FLUCTUATING (Fragile). Target Strategy: KEEP STABLE STREAKS.")
        keep_stable = True
        
    print(f"Optimal Noise Selected: STD = {best_std}\n---------------------------------------------\n")
    return best_std, keep_stable

# ==========================================
# 4. Main Pipeline Execution
# ==========================================

def run_debris_removal_pipeline():
    print("Initializing Detectron2 predictor...")
    predictor = setup_detectron_predictor(MODEL_WEIGHTS_PATH)
    
    print(f"Loading raw submission file from {RAW_SUBMISSION_CSV}...")
    df = pd.read_csv(RAW_SUBMISSION_CSV)
    
    # 1. Run Calibration
    calibrated_std, keep_stable = calibrate_pipeline(df, predictor, num_samples=5)
    
    # 2. Main Run Loop
    print("Beginning full dataset cleanup execution...")
    cleaned_rows = []
    delta_thresh = 0.3
    iou_thresh = 0.5
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Purging Debris"):
        image_id = row['image_id']
        pred_string = row['prediction_string']
        
        orig_streaks = parse_prediction_string(pred_string)
        
        if not orig_streaks:
            cleaned_rows.append({"id": row['id'], "image_id": image_id, "prediction_string": " "})
            continue
            
        img_path = Path(TEST_IMAGE_DIR) / f"{image_id}.png"
        image = load_for_inference(img_path)
        
        if image is None:
            cleaned_rows.append({"id": row['id'], "image_id": image_id, "prediction_string": pred_string})
            continue
            
        # Add the calibrated amount of noise
        noisy_image = add_gaussian_noise(image, std=calibrated_std) 
        
        outputs = predictor(noisy_image)["instances"].to("cpu")
        noisy_boxes = outputs.pred_boxes.tensor.numpy()
        noisy_scores = outputs.scores.numpy()
        new_streaks = [{'bbox': b, 'conf': s} for b, s in zip(noisy_boxes, noisy_scores)]
        
        robust_streaks_strings = []
        
        for orig in orig_streaks:
            orig_box = orig['bbox']
            orig_conf = orig['conf']
            
            best_iou = 0.0
            matched_conf = 0.0 
            
            for new_strk in new_streaks:
                iou = calculate_iou(orig_box, new_strk['bbox'])
                if iou > best_iou:
                    best_iou = iou
                    matched_conf = new_strk['conf']
            
            if best_iou < iou_thresh:
                matched_conf = 0.0
                
            conf_delta = abs(orig_conf - matched_conf)
            
            # Conditional evaluation based on the calibration phase's choice
            if keep_stable:
                # Normal behavior: Keep if confidence stays close to original
                should_keep = (conf_delta <= delta_thresh)
            else:
                # Inverted behavior: Keep if confidence fluctuates wildly (because poison is locked in place)
                should_keep = (conf_delta > delta_thresh)
                
            if should_keep:
                x1, y1, x2, y2 = orig_box
                w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
                robust_streaks_strings.extend([f"{orig_conf:.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])
        
        final_str = " ".join(robust_streaks_strings) if robust_streaks_strings else " "
        cleaned_rows.append({
            "id": row['id'],
            "image_id": image_id,
            "prediction_string": final_str
        })

    output_df = pd.DataFrame(cleaned_rows)
    output_df.to_csv(CLEANED_OUTPUT_CSV, index=False)
    print(f"\nProcessing complete! Cleaned file saved at: {CLEANED_OUTPUT_CSV}")

if __name__ == "__main__":
    run_debris_removal_pipeline()